In [1]:
pip install wandb   

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install --upgrade wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 49.0 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.22.2
    Uninstalling wandb-0.22.2:
      Successfully uninstalled wandb-0.22.2


In [3]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WAND_API")

# This will now accept the 86-character key
wandb.login(key=wandb_key, relogin=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: safalnarshing (safalnarshing-kathmandu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
!git clone https://github.com/NVIDIA/NeMo.git 

Cloning into 'NeMo'...
remote: Enumerating objects: 264643, done.
remote: Counting objects: 100% (669/669), done.
remote: Compressing objects: 100% (401/401), done.
remote: Total 264643 (delta 493), reused 268 (delta 268), pack-reused 263974 (from 5)
Receiving objects: 100% (264643/264643), 473.53 MiB | 38.11 MiB/s, done.
Resolving deltas: 100% (200441/200441), done.


In [6]:
pip install pybind11

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install wheel setuptools pip --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.3 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.45.1
    Uninstalling wheel-0.45.1:
      Successfully uninstalled wheel-0.45.1
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
fastai 2.8.4 requi

In [8]:
pip install fasttext

Note: you may need to restart the kernel to use updated packages.


In [9]:
!pip install nemo_toolkit['asr']==2.1.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of pyannote-metrics to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 15.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 15.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━

In [10]:
!pip install sentencepiece

In [11]:
import json
import os
import soundfile as sf
import numpy as np
from tqdm import tqdm

# 1. Setup Directories
# Ensure this matches exactly what you see in the 'Data' sidebar
INPUT_DIR = "/kaggle/input/datasets/technohubzeke/final-data"
OUTPUT_DIR = "/kaggle/working/mono_audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

manifest_files = ["nemo_train.json", "nemo_val.json", "nemo_test.json"]

def process_manifest(manifest_filename):
    input_path = os.path.join(INPUT_DIR, manifest_filename)
    output_manifest_path = os.path.join("/kaggle/working", f"mono_{manifest_filename}")
    
    if not os.path.exists(input_path):
        print(f"Skipping {manifest_filename} (File not found at {input_path})")
        return

    new_entries = []
    print(f"Processing {manifest_filename}...")
    
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f):
            entry = json.loads(line)
            
            # --- THE FIX STARTS HERE ---
            # Extract ONLY the filename (e.g., 'NEW0004.wav')
            filename = os.path.basename(entry['audio_filepath'])
            
            # Reconstruct the path to look specifically in the 'audio' or 'wav' folder
            # If your files are in a subfolder, change 'audio' to 'wav' or whatever applies
            full_audio_path = os.path.join(INPUT_DIR, "audio", "audio", filename) 
            
            # Fallback: If the path above is wrong, search the whole INPUT_DIR
            if not os.path.exists(full_audio_path):
                # This searches all subfolders for the file if the hardcoded path above fails
                import glob
                found = glob.glob(f"{INPUT_DIR}/**/{filename}", recursive=True)
                if found:
                    full_audio_path = found[0]
                else:
                    print(f"File not found anywhere in INPUT_DIR: {filename}")
                    continue
            # --- THE FIX ENDS HERE ---

            new_audio_path = os.path.join(OUTPUT_DIR, filename)
            
            try:
                data, samplerate = sf.read(full_audio_path)
                
                if len(data.shape) > 1 and data.shape[1] > 1:
                    data = data.mean(axis=1)
                
                sf.write(new_audio_path, data, samplerate)
                
                entry['audio_filepath'] = new_audio_path
                new_entries.append(entry)
                
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    with open(output_manifest_path, 'w', encoding='utf-8') as f_out:
        for entry in new_entries:
            f_out.write(json.dumps(entry) + '\n')
            
    print(f"Finished. Saved new manifest to: {output_manifest_path}")

for manifest in manifest_files:
    process_manifest(manifest)

Processing nemo_train.json...


4575it [01:34, 48.54it/s]


Finished. Saved new manifest to: /kaggle/working/mono_nemo_train.json
Processing nemo_val.json...


576it [00:11, 52.13it/s]


Finished. Saved new manifest to: /kaggle/working/mono_nemo_val.json
Processing nemo_test.json...


576it [00:10, 53.70it/s]

Finished. Saved new manifest to: /kaggle/working/mono_nemo_test.json


In [12]:
import json

manifest_path = "/kaggle/working/mono_nemo_train.json"

with open(manifest_path, 'r') as f:
    first_line = f.readline()
    print(f"Raw first line: {first_line}")
    try:
        data = json.loads(first_line)
        print(f"Parsed JSON: {data}")
        print(f"Text field content: {data.get('text')}")
    except Exception as e:
        print(f"Error parsing JSON: {e}")

Raw first line: {"audio_filepath": "/kaggle/working/mono_audio/NEW0004.wav", "duration": 3.5627, "text": "\u0925\u094d\u0935 \u0925\u093e\u0938\u092f\u094d \u091b\u094d\u092f\u0932\u093f\u0917\u0941 \u092e\u0942 \u092d\u093e\u092f\u094d \u0924\u0947\u0932\u0941\u0917\u0941 \u092d\u093e\u0937\u093e \u0916"}

Parsed JSON: {'audio_filepath': '/kaggle/working/mono_audio/NEW0004.wav', 'duration': 3.5627, 'text': 'थ्व थासय् छ्यलिगु मू भाय् तेलुगु भाषा ख'}
Text field content: थ्व थासय् छ्यलिगु मू भाय् तेलुगु भाषा ख


In [14]:
import json

manifest_path = "/kaggle/working/mono_nemo_train.json"

with open(manifest_path, 'r') as f:
    first_line = f.readline()
    print(f"Raw first line: {first_line}")
    try:
        data = json.loads(first_line)
        print(f"Parsed JSON: {data}")
        print(f"Text field content: {data.get('text')}")
    except Exception as e:
        print(f"Error parsing JSON: {e}")

Raw first line: {"audio_filepath": "/kaggle/working/mono_audio/NEW0004.wav", "duration": 3.5627, "text": "\u0925\u094d\u0935 \u0925\u093e\u0938\u092f\u094d \u091b\u094d\u092f\u0932\u093f\u0917\u0941 \u092e\u0942 \u092d\u093e\u092f\u094d \u0924\u0947\u0932\u0941\u0917\u0941 \u092d\u093e\u0937\u093e \u0916"}

Parsed JSON: {'audio_filepath': '/kaggle/working/mono_audio/NEW0004.wav', 'duration': 3.5627, 'text': 'थ्व थासय् छ्यलिगु मू भाय् तेलुगु भाषा ख'}
Text field content: थ्व थासय् छ्यलिगु मू भाय् तेलुगु भाषा ख


In [15]:
!python NeMo/scripts/tokenizers/process_asr_text_tokenizer.py \
    --manifest "/kaggle/working/mono_nemo_train.json" \
    --data_root "tokens" \
    --vocab_size 128 \
    --tokenizer "spe" \
    --spe_type "unigram" \
    --spe_max_sentencepiece_length 200 \
    --spe_character_coverage 1.0 \
    --log


INFO:root:Finished extracting manifest : /kaggle/working/mono_nemo_train.json
INFO:root:Finished extracting all manifests ! Number of sentences : 4575
[NeMo I 2026-02-15 16:25:17 nemo_logging:393] Processing tokens/text_corpus/document.txt and store at tokens/tokenizer_spe_unigram_v128_max_200
sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=tokens/text_corpus/document.txt --model_prefix=tokens/tokenizer_spe_unigram_v128_max_200/tokenizer --vocab_size=128 --shuffle_input_sentence=true --hard_vocab_limit=false --model_type=unigram --character_coverage=1.0 --bos_id=-1 --eos_id=-1 --normalization_rule_name=nmt_nfkc_cf --max_sentencepiece_length=200 --remove_extra_whitespaces=false
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: tokens/text_corpus/document.txt
  input_format: 
  model_prefix: tokens/tokenizer_spe_unigram_v128_max_200/tokenizer
  model_type: UNIGRAM
  vocab_size: 128
  self_test_sample_size: 0
  character_coverage: 1
  

In [16]:
import yaml

config = """
name: "Conformer-CTC-BPE-Finetune"

init_from_ptl_ckpt: "/kaggle/input/datasets/jennypoudel/newari-data/Conformer-CTC-BPE--val_wer0.2177-epoch49.ckpt"

model:
  sample_rate: 16000
  log_prediction: true
  ctc_reduction: 'mean_batch'
  skip_nan_grad: false
  use_cer: true

  train_ds:
    manifest_filepath: "/kaggle/working/mono_nemo_train.json"
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 16.7
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: "synced_randomized"
    bucketing_batch_size: null

  validation_ds:
    manifest_filepath: "/kaggle/working/mono_nemo_val.json"
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: false
    use_start_end_token: false
    num_workers: 4
    pin_memory: true

  test_ds:
    manifest_filepath: "/kaggle/working/mono_nemo_test.json"
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: false
    use_start_end_token: false
    num_workers: 4
    pin_memory: true

  tokenizer:
    dir: tokens/tokenizer_spe_unigram_v128_max_200
    type: bpe

  preprocessor:
    _target_: nemo.collections.asr.modules.AudioToMelSpectrogramPreprocessor
    sample_rate: ${model.sample_rate}
    normalize: "per_feature"
    window_size: 0.025
    window_stride: 0.01
    window: "hann"
    features: 80
    n_fft: 512
    log: true
    frame_splicing: 1
    dither: 0.00001
    pad_to: 0
    pad_value: 0.0

  # ---------------------------------------------------------------------
  # SPECAUGMENT: Switchboard Strong Policy (Corrected)
  # ---------------------------------------------------------------------
  spec_augment:
    _target_: nemo.collections.asr.modules.SpectrogramAugmentation
    freq_masks: 2
    time_masks: 2
    freq_width: 27
    time_width: 70

  encoder:
    _target_: nemo.collections.asr.modules.ConformerEncoder
    feat_in: ${model.preprocessor.features}
    feat_out: -1
    n_layers: 18
    d_model: 256
    subsampling: striding
    subsampling_factor: 4
    subsampling_conv_channels: -1
    causal_downsampling: false
    ff_expansion_factor: 4
    self_attention_model: rel_pos
    n_heads: 4
    att_context_size: [-1, -1]
    att_context_style: regular
    xscaling: true
    untie_biases: true
    pos_emb_max_len: 5000
    conv_kernel_size: 31
    conv_norm_type: 'batch_norm'
    conv_context_size: null
    dropout: 0.1
    dropout_pre_encoder: 0.1
    dropout_emb: 0.0
    dropout_att: 0.1
    stochastic_depth_drop_prob: 0.0
    stochastic_depth_mode: linear
    stochastic_depth_start_layer: 1

  decoder:
    _target_: nemo.collections.asr.modules.ConvASRDecoder
    feat_in: null
    num_classes: -1
    vocabulary: []

  interctc:
    loss_weights: []
    apply_at_layers: []

  optim:
    name: adamw
    lr: 0.00005
    betas: [0.9, 0.98]
    weight_decay: 1e-3
    sched:
      name: CosineAnnealing
      min_lr: 1e-6
      warmup_steps: 3000

trainer:
  devices: -1
  num_nodes: 1
  max_epochs: 100
  max_steps: -1
  val_check_interval: 1.0
  accelerator: auto
  accumulate_grad_batches: 8
  gradient_clip_val: 1.0
  precision: 32
  log_every_n_steps: 100
  enable_progress_bar: True
  num_sanity_val_steps: 0
  check_val_every_n_epoch: 1
  sync_batchnorm: true
  enable_checkpointing: False
  logger: false
  benchmark: false
  max_time: "00:11:30:00"

exp_manager:
  exp_dir: null
  name: ${name}
  create_tensorboard_logger: true
  create_checkpoint_callback: true
  checkpoint_callback_params:
    monitor: "val_wer"
    mode: "min"
    save_top_k: 3
    always_save_nemo: True
  
  # ---------------------------------------------------------
  # CHANGE: Early Stopping DISABLED to force 100 epochs
  # ---------------------------------------------------------
  create_early_stopping_callback: False
   
  resume_if_exists: false
  resume_ignore_no_checkpoint: false
  create_wandb_logger: True
  wandb_logger_kwargs:
    name: "finetune-newari-asr-final-0.0005"
    project: "Conformer-Finetuning"
"""

with open('finetune_config.yaml', 'w') as file:
    file.write(config)

print("finetune_config.yaml updated: SpecAugment ENABLED, Early Stopping DISABLED.")

finetune_config.yaml updated: SpecAugment ENABLED, Early Stopping DISABLED.


In [ ]:
file_path = "NeMo/examples/asr/asr_ctc/speech_to_text_ctc_bpe.py"

# The new content with BOTH PyTorch and NumPy fixes + Encoder Freezing
new_content = '''
# Copyright (c) 2020, NVIDIA CORPORATION.  All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import lightning.pytorch as pl
from omegaconf import OmegaConf
import torch 
import numpy as np

# ===================================================================
#  FIX 1: NUMPY 2.0 COMPATIBILITY PATCH
#  NeMo < 2.0 uses 'np.sctypes' which was removed in NumPy 2.0
# ===================================================================
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'float': [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others': [bool, object, bytes, str, np.void]
    }

# ===================================================================
#  FIX 2: PYTORCH 2.6+ "WEIGHTS ONLY" LOAD ERROR
#  Override torch.load to disable security checks for NeMo checkpoints
# ===================================================================
_original_torch_load = torch.load

def _custom_torch_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

torch.load = _custom_torch_load
# ===================================================================

from nemo.collections.asr.models.ctc_bpe_models import EncDecCTCModelBPE
from nemo.core.config import hydra_runner
from nemo.utils import logging
from nemo.utils.exp_manager import exp_manager

import sys
import importlib.util

# Path to trainer_utils.py
file_path = "NeMo/nemo/utils/trainer_utils.py"

# Load the module from the file path
spec = importlib.util.spec_from_file_location("trainer_utils", file_path)
trainer_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer_utils)

# Now you can use resolve_trainer_cfg
resolve_trainer_cfg = trainer_utils.resolve_trainer_cfg

# ===================================================================
#  ENCODER FREEZING FUNCTION (ALL LAYERS FROZEN)
# ===================================================================
def freeze_encoder_layers(model):
    """
    Freeze ALL encoder layers for transfer learning.
    Only the decoder head is trainable for new vocabulary.
    
    Conformer Structure:
    ├── Preprocessor        ❄️  Frozen
    ├── Spec Augmentation   ❄️  Frozen
    ├── Encoder             ❄️  ALL LAYERS FROZEN
    │   ├── Layer 0         ❄️  Frozen
    │   ├── Layer 1         ❄️  Frozen
    │   ├── Layer 2         ❄️  Frozen
    │   └── ...
    │   └── Layer 16        ❄️  Frozen (all encoder layers)
    └── Decoder             🔓 Trainable (new vocabulary)
    """
    print("\\n" + "="*60)
    print("FREEZING ALL ENCODER LAYERS")
    print("="*60)
    
    # 1. Freeze Preprocessor
    if hasattr(model, 'preprocessor'):
        for param in model.preprocessor.parameters():
            param.requires_grad = False
        print("❄️  Frozen: Preprocessor")
    
    # 2. Freeze Spec Augmentation
    if hasattr(model, 'spec_augmentation'):
        for param in model.spec_augmentation.parameters():
            param.requires_grad = False
        print("❄️  Frozen: Spec Augmentation")
    
    # 3. Freeze ALL encoder layers
    if hasattr(model, 'encoder') and hasattr(model.encoder, 'layers'):
        total_layers = len(model.encoder.layers)
        print(f"\\nFreezing all {total_layers} encoder layers:")
        
        for i, layer in enumerate(model.encoder.layers):
            for param in layer.parameters():
                param.requires_grad = False
            print(f"❄️  Frozen: Encoder Layer {i}")
    
    # 4. Keep Decoder trainable (new vocabulary)
    if hasattr(model, 'decoder'):
        for param in model.decoder.parameters():
            param.requires_grad = True
        print("\\n🔓 Trainable: Decoder (new vocabulary only)")
    
    # Count trainable parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    
    print("\\n" + "="*60)
    print("PARAMETER SUMMARY")
    print("="*60)
    print(f"Total Parameters:     {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
    print(f"Frozen Parameters:    {frozen_params:,} ({frozen_params/total_params*100:.2f}%)")
    print("="*60 + "\\n")

@hydra_runner(config_path="../conf/citrinet/", config_name="config_bpe")
def main(cfg):
    logging.info(f'Hydra config: {OmegaConf.to_yaml(cfg)}')

    trainer = pl.Trainer(**resolve_trainer_cfg(cfg.trainer))
    exp_manager(trainer, cfg.get("exp_manager", None))
    asr_model = EncDecCTCModelBPE(cfg=cfg.model, trainer=trainer)

    # Initialize the weights of the model from another model, if provided via config
    asr_model.maybe_init_from_pretrained_checkpoint(cfg)
    
    # ===================================================================
    #  APPLY ENCODER FREEZING - ALL LAYERS
    # ===================================================================
    freeze_encoder_layers(asr_model)

    trainer.fit(asr_model)

    if hasattr(cfg.model, 'test_ds') and cfg.model.test_ds.manifest_filepath is not None:
        if asr_model.prepare_test(trainer):
            trainer.test(asr_model)


if __name__ == '__main__':
    main()

'''

# Open the file in write mode and overwrite the content
with open(file_path, "w") as file:
    file.write(new_content)
    
print(f"File {file_path} has been updated with Encoder Freezing + PyTorch + NumPy fixes.")
print("\\n✅ Updated Encoder Freezing Configuration:")
print("   - Preprocessor: FROZEN")
print("   - Spec Augmentation: FROZEN")
print("   - ALL Encoder Layers (0-16): FROZEN ❄️")
print("   - Decoder: TRAINABLE (new vocabulary only) 🔓")

File NeMo/examples/asr/asr_ctc/speech_to_text_ctc_bpe.py has been updated with Encoder Freezing + PyTorch + NumPy fixes.
\n✅ Encoder Freezing Configuration:
   - Layers 0-12: FROZEN (general speech features)
   - Layers 13-16: TRAINABLE (language-specific adaptation)
   - Preprocessor: FROZEN
   - Spec Augmentation: FROZEN
   - Decoder: TRAINABLE (new vocabulary)


In [18]:
!python /kaggle/working/NeMo/examples/asr/asr_ctc/speech_to_text_ctc_bpe.py \
  --config-path="/kaggle/working/" \
  --config-name="finetune_config" \
  ++init_from_ptl_ckpt="/kaggle/input/datasets/jennypoudel/newari-data/Conformer-CTC-BPE--val_wer0.2177-epoch49.ckpt" \
  model.train_ds.manifest_filepath="/kaggle/working/mono_nemo_train.json" \
  model.validation_ds.manifest_filepath="/kaggle/working/mono_nemo_val.json" \
  model.test_ds.manifest_filepath="/kaggle/working/mono_nemo_test.json" \
  model.tokenizer.dir="tokens/tokenizer_spe_unigram_v128_max_200"

[NeMo W 2026-02-15 16:25:28 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-02-15 16:25:28 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-02-15 16:25:28 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-02-15 16:25:28 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    
[NeMo I 2026-02-15 16:25:32 nemo_logging:393] Hydra config: name: Conformer-CTC-BPE-Finetune
    init_from_ptl_ckpt: /kaggle/input/